In [1]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import os
import gc
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from detection_baselines.detection_stats import load_stats_dataset, extract_stats
from egh_vlm.utils import Qwen3ModelBundle, load_phd_dataset

In [2]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/stats_phd_base_qwen3.pt'

#### Set Up

In [3]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
    sample_size=1000)

features = load_stats_dataset(FEATURES_PATH) if os.path.isfile(FEATURES_PATH) else None
print(f'Length of processed features: {len(features) if features is not None else 0}')

Successfully load the PhD dataset with: 1000 samples.
Length of processed features: 20


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype=torch.float16,
    device_map=device
)
print("config.dtype:", model.config.dtype)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 1024)
model_bundle = Qwen3ModelBundle(model, processor, device)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

config.dtype: torch.float16


#### Experiment

In [5]:
stats = extract_stats(
    dataset=dataset,
    model_bundle=model_bundle,
    client_type='qwen3',
    save_path=FEATURES_PATH,
    mask_mode=None,
)
if len(stats) > 0:
    print('Mean Entropy:', stats[0][1])
    print('Max Entropy:', stats[0][2])
    print('Mean Prob:', stats[0][3])
    print('Max Prob:', stats[0][4])

Extracting stats for client qwen3: 100%|██████████| 1000/1000 [07:11<00:00,  2.32it/s]

Mean Entropy: tensor([1.0689])
Max Entropy: tensor([3.6832])
Mean Prob: tensor([0.6982])
Max Prob: tensor([1.])


In [6]:
# Clean up GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()